
# BiLSTM vs. MolFormer — Encoder Comparison

This notebook compares two SMILES-based molecular encoders on the BACE-1 bioactivity prediction task:

- **BiLSTM:** A trainable bidirectional LSTM encoder.
- **MolFormer:** A pretrained transformer encoder with frozen weights and a trainable regression head.




Both encoders are compared using the same BACE-1 dataset, target variable, and scaffold split, verified through dataset and split hashes. The comparison uses recorded validation results from previously completed hyperparameter sweeps. Configurations are ranked by mean validation RMSE across three random seeds. Test-set evaluation is outside the scope of this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# locate the AI directory regardless of where Jupyter starts
cwd = Path.cwd().resolve()

if (cwd / "ai" / "outputs").is_dir():
    ROOT = cwd / "ai"
elif (cwd / "outputs").is_dir():
    ROOT = cwd
elif (cwd.parent / "outputs").is_dir():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate ai/outputs")

bilstm = pd.read_csv(
    ROOT / "outputs/encoders/BiLSTM/sweep/3-finalists.csv"
)

molformer = pd.read_csv(
    ROOT / "outputs/encoders/SMILES/1-head.csv"
)

print("BiLSTM runs:", len(bilstm))
print("MolFormer runs:", len(molformer))

In [ ]:
# verify that both encoders used the same dataset and split
for column in ("data_sha256", "split_sha256"):
    assert column in bilstm.columns and column in molformer.columns, (
        f"Missing {column} in encoder results"
    )

    bilstm_values = bilstm[column].dropna().unique()
    molformer_values = molformer[column].dropna().unique()

    assert bilstm[column].notna().all() and molformer[column].notna().all(), (
        f"Missing values in {column}"
    )
    assert len(bilstm_values) == len(molformer_values) == 1, (
        f"Inconsistent {column} values"
    )
    assert bilstm_values[0] == molformer_values[0], (
        f"BiLSTM and MolFormer have different {column} values"
    )

print("Dataset hashes match!")
print("Split hashes match!")

In [ ]:
import json

# exact finalists from bilstm_encoder_sweep.py
expected_configs = {
    (128, 128, 2, 0.2, 1e-3, 64),
    (64, 64, 2, 0.2, 1e-3, 64),
    (64, 256, 1, 0.2, 1e-3, 128),
}

config_keys = [
    "EMBEDDING_DIM",
    "HIDDEN_DIM",
    "NUM_LAYERS",
    "DROPOUT",
    "LR",
    "BATCH_SIZE",
]

def config_identity(cfg):
    params = json.loads(cfg)
    return tuple(params[key] for key in config_keys)

# exclude custom configurations from the default ranking
bilstm_finalists = bilstm[
    bilstm["cfg"].map(config_identity).isin(expected_configs)
].copy()

found_configs = set(bilstm_finalists["cfg"].map(config_identity))

if found_configs != expected_configs:
    raise ValueError("Missing one or more expected BiLSTM finalists")

# require exactly one run for each seed per finalist
for cfg, group in bilstm_finalists.groupby("cfg"):
    if len(group) != 3 or set(group["seed"]) != {0, 1, 2}:
        raise ValueError(
            "Each BiLSTM finalist must have exactly seeds 0, 1, and 2"
        )

bilstm_summary = (
    bilstm_finalists.groupby("cfg")
    .agg(
        mean_RMSE=("val_RMSE", "mean"),
        std_RMSE=("val_RMSE", "std"),
        mean_R2=("val_R2", "mean"),
        seeds=("seed", "nunique")
    )
    .sort_values("mean_RMSE")
)

# molFormer: compare the three learning rates
molformer_summary = (
    molformer.groupby("LR")
    .agg(
        mean_RMSE=("val_RMSE", "mean"),
        std_RMSE=("val_RMSE", "std"),
        mean_R2=("val_R2", "mean"),
        seeds=("seed", "nunique")
    )
    .sort_values("mean_RMSE")
)

print("Best BiLSTM configurations:")
display(bilstm_summary.round(4))

print("MolFormer learning rate comparison:")
display(molformer_summary.round(4))

In [ ]:
comparison = pd.DataFrame({
    "Encoder": ["BiLSTM", "MolFormer"],
    "Mean Val RMSE": [
        bilstm_summary.iloc[0]["mean_RMSE"],
        molformer_summary.iloc[0]["mean_RMSE"]
    ],
    "Std Val RMSE": [
        bilstm_summary.iloc[0]["std_RMSE"],
        molformer_summary.iloc[0]["std_RMSE"]
    ],
    "Mean Val R²": [
        bilstm_summary.iloc[0]["mean_R2"],
        molformer_summary.iloc[0]["mean_R2"]
    ]
})

display(comparison.round(4))

improvement = (
    (comparison.iloc[0]["Mean Val RMSE"] - comparison.iloc[1]["Mean Val RMSE"])
    / comparison.iloc[0]["Mean Val RMSE"]
    * 100
)

print(f"MolFormer RMSE improvement: {improvement:.2f}%")

In [ ]:
plt.figure(figsize=(7, 4))

plt.bar(
    comparison["Encoder"],
    comparison["Mean Val RMSE"],
    yerr=comparison["Std Val RMSE"],
    capsize=6,
    color=["steelblue", "darkorange"]
)

plt.ylabel("Mean Validation RMSE")
plt.title("BiLSTM vs. MolFormer on BACE-1")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Conclusion

Compared two SMILES-based molecular encoders on the BACE-1 benchmark, selecting each encoder's best hyperparameter configuration based on mean validation RMSE across three random seeds.

- **BiLSTM:** Mean validation RMSE = 0.8859 ± 0.0287, with mean R² = 0.5688.
- **MolFormer:** Mean validation RMSE = 0.8718 ± 0.0010, with mean R² = 0.5827.

MolFormer achieved approximately **1.59% lower validation RMSE** and exhibited less variation across seeds.

These results suggest that frozen pretrained MolFormer embeddings with a linear prediction head provide a modest improvement over the trained BiLSTM encoder under the evaluated configurations.